# Business Analyst Agents

## Setup

In [1]:
llm_config = {"model": "gemini-3.1-flash-lite","api_key":"","api_type":"google"}

## The task!

In [2]:
task = '''
        Feature Overview
•
Feature Title: Save for Later (Shopping Cart Enhancement)
•
Target Release: Q3 2026
•
Objective: Allow users to move items out of their active shopping cart into a "Saved for Later" list. This reduces cart abandonment and drives future sales.
Functional Requirements
1.
Cart Page Integration:
o
Every item in the shopping cart must display a "Save for Later" button next to the "Remove" button.
o
Clicking "Save for Later" must instantly move the item out of the active cart and place it into a new section below the cart titled "Saved for Later".
o
The active cart total price and item count must automatically update without reloading the page.
2.
Saved for Later Section:
o
This section only appears if it contains at least one item.
o
Each item in this list must display the product image, title, current price, and stock status (e.g., "In Stock", "Only 2 Left").
o
Each item must have a "Move to Cart" button and a "Delete" button.
3.
Item Management:
o
Clicking "Move to Cart" must move the item back to the active cart. If the item is out of stock, show a warning and do not move it.
o
Clicking "Delete" must permanently remove the item from the list.
4.
User State:
o
Guest users can save items, but their list is only saved in their local browser session.
o
Logged-in users must have their list synced to their account database so they can view it across multiple devices.
       '''


In [3]:
import autogen
from autogen.retrieve_utils import TEXT_FORMATS,extract_text_from_pdf
from autogen.coding import  CodeBlock, LocalCommandLineCodeExecutor
from pathlib import Path
pdf_files = ["BRD.pdf"]
docs_content= extract_text_from_pdf("BRD.pdf")

In [4]:
work_dir = Path("AC_Stories")
work_dir.mkdir(exist_ok=True)
executor = LocalCommandLineCodeExecutor(work_dir=work_dir)

In [5]:
# from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent
# ragproxyagent = RetrieveUserProxyAgent(
#     name="ragproxyagent",
#     human_input_mode="NEVER",
#     max_consecutive_auto_reply=1,
#     retrieve_config={
#         "task":"qa",
#         "docs_path": ["BA_Agent/BRD.pdf"],
#         "docs_content": docs_content,  # Provide the extracted content directly
#         "chunk_token_size": 2000,
#         "model": "gemini-3.1-flash-lite",
#         "vector_db": "chroma",
#         "overwrite": True,
#         "get_or_create": True,
#     },
#     code_execution_config={
#         "executor": executor,
#     },
# )

In [6]:
# message=ragproxyagent.message_generator
# message

# BA Manager Agent to provide the BRD document and control the flow

In [7]:
BA_Manager_Agent = autogen.AssistantAgent(
    name="BA_Manager_Agent",
    human_input_mode="NEVER",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a BA Manager. You assign the task to each agent to read the requirement document."
                " Extract the features in the requirement document "
                "Create user stories based on the extracted features "
                "Write the acceptance criteria for each user stories."
                "Finally you review the acceptance criteria inline with the requirements",
)

# Extract features from BRD

In [8]:
from autogen import AssistantAgent
from autogen import initiate_chats

feature_extract_agent = autogen.AssistantAgent(
    name="feature_extract_agent",
    system_message="You are a Feature extract agent. Extract distinct, atomic features from raw requirements documents BRD.pdf. " 
        "Expert at parsing messy text and identifying core business logic.",
    llm_config=llm_config,
)

# Convert Features into user stories

In [9]:
User_story_agent = autogen.AssistantAgent(
    name="User_story_agent",
    llm_config=llm_config,
    system_message="You are User story creation agent "
        "Create the user stories based on the extracted features from the BRD document,"
        "Master of the standard format: 'As a [user], I want [action] so that [benefit]'."
        "Ensure all the requirements mentioned in the BRD are covered in the user stories"
        "concrete and to the point."
        "Begin the creation by stating your role.",
)

# Create Acceptance Criteria for each user stories

In [10]:
AC_agent = autogen.AssistantAgent(
    name="AC_agent",
    llm_config=llm_config,
    system_message="You are a Acceptance Criteria Agent "
        "You are ability to create acceptance criteria for each user story"
        "For each story, provide 2 detailed Acceptance Criteria using Given-When-Then syntax"
        "You ensure that each point is covered in the user story and inline with the BRD document."
        "concrete and to the point. "
        "Begin the creation by stating your role.",
)

In [11]:
# res = BA_Manager_Agent.initiate_chat(
#     recipient=feature_extract_agent,
#     message=task,
#     max_turns=2,
#     summary_method="last_msg"
# )

In [12]:
# qa_problem = """You are BA Manager. You assign the task to each agent to read the requirement document.
#                 Extract the features in the requirement document
#                 Create user stories based on the extracted features
#                 Write the acceptance criteria for each user stories.
#                 Finally you review the acceptance criteria inline with the requirements"""
# chat_result = ragproxyagent.initiate_chat(feature_extract_agent, message=ragproxyagent.message_generator, problem=qa_problem,max_turns=2)

# Nested Chat to solve the task

In [13]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

BA_chats = [
    # {
    #  "recipient": ragproxyagent, 
    #  "message": reflection_message, 
    #  "summary_method": "reflection_with_llm",
    #  # "summary_args": {"summary_prompt" : 
    #  #    "Return features into as JSON object only:"
    #  #    "{'Agent': '', 'Features': ''}. Here Agent should be your role",},
    #  "max_turns": 1},
    {
     "recipient": feature_extract_agent, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return features into as JSON object only:"
        "{'Agent': '', 'Features': ''}. Here Agent should be your role",},
     "max_turns": 1},
    {
    "recipient": User_story_agent, 
    "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return User stories into as JSON object only:"
        "{'Agent': '', 'User Stories': ''}.Here Agent should be your role",},
     "max_turns": 1},
    {"recipient": AC_agent, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return Acceptance Criteria for each story into as JSON object only:"
        "{'Agent': '', 'AC': ''}.Here Agent should be your role",},
     "max_turns": 1},     
]


In [16]:
BA_Manager_Agent.register_nested_chats(
    BA_chats,
    trigger=feature_extract_agent,
)

In [17]:
res = BA_Manager_Agent.initiate_chat(
    recipient=feature_extract_agent,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

BA_Manager_Agent (to feature_extract_agent):


        Feature Overview
•
Feature Title: Save for Later (Shopping Cart Enhancement)
•
Target Release: Q3 2026
•
Objective: Allow users to move items out of their active shopping cart into a "Saved for Later" list. This reduces cart abandonment and drives future sales.
Functional Requirements
1.
Cart Page Integration:
o
Every item in the shopping cart must display a "Save for Later" button next to the "Remove" button.
o
Clicking "Save for Later" must instantly move the item out of the active cart and place it into a new section below the cart titled "Saved for Later".
o
The active cart total price and item count must automatically update without reloading the page.
2.
Saved for Later Section:
o
This section only appears if it contains at least one item.
o
Each item in this list must display the product image, title, current price, and stock status (e.g., "In Stock", "Only 2 Left").
o
Each item must have a "Move to Cart" button and a "Delet

In [22]:
AC_agent_chat = BA_Manager_Agent.chat_messages[AC_agent]
detailed_output = ""
for message in reversed(AC_agent_chat):
    if message.get("role") == "assistant": 
        # Skip messages sent *to* the PO, we only want responses *from* the PO
        continue
    
    content = message.get("content", "")
    if content:
        detailed_output = content
        break

if "TERMINATE" in detailed_output:
    detailed_output = detailed_output.replace("TERMINATE", "").strip()

# Create and write to the text document
output_filename = "AC_Stories/final_user_stories_and_ac.txt"

with open(output_filename, "w", encoding="utf-8") as file:
    file.write("# 📋 Generated User Stories and Acceptance Criteria\n")
    file.write("==================================================\n\n")
    file.write(detailed_output)
    
print(f"🎉 Success! The document has been updated and saved to: {output_filename}")


🎉 Success! The document has been updated and saved to: AC_Stories/final_user_stories_and_ac.txt


In [ ]:
# chat_result = ragproxyagent.initiate_chat(BA_Manager_Agent, message=ragproxyagent.message_generator, problem=task)

In [21]:
detailed_output

'I am the Acceptance Criteria Agent. I have reviewed the user stories provided and have developed the following acceptance criteria based on your requirements.\n\n---\n\n### **[FEAT-001] "Save for Later" Action Button**\n*   **AC1:** Given I am viewing my active shopping cart, when I locate an item, then the "Save for Later" button must be clearly visible and positioned immediately adjacent to the "Remove" button.\n*   **AC2:** Given I click the "Save for Later" button on an item, when the action completes, then the item must be successfully removed from the active cart view.\n\n### **[FEAT-002] Dynamic Cart Updates**\n*   **AC1:** Given I have items in my cart, when I move an item to "Saved for Later", then the total price and item count display must update instantly without the browser reloading the page.\n*   **AC2:** Given I move an item, when the UI updates, then the cart total and count must reflect the correct arithmetic sum of remaining active items.\n\n### **[FEAT-003] Conditi

## Get the summary